#**Tugas 1 - Scraping Berita Detikcom**

**Install Library**

In [1]:
pip install requests beautifulsoup4 pandas openpyxl tqdm trafilatura

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 23.7 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 3.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 20.5 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 24.9 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.4/801.4 kB 11.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [trafilatura] [trafilatura]
Note: you may need to restart the kernel to use updated packages.


Code diatas berfungsi untuk menginstall pustaka pustaka pyhton yang dibutuhkan ke dalam lingkungan kerja seperti google colab, mencakup request untuk mengirim permintaan HTTP, beautifulsoup4 dan trafilatura untuk scraping serta ekstraksi konten web, pandas dan openpyxl untuk pengolahan data serta pembuatan file excel, serta tqdm untuk menampilkan indikator proses.

**Ambil Isi Artikel Lengkap Menggunakan trafilatura (tqdm + Paginasi)**

In [3]:
import re
import time
import pandas as pd
import requests
import trafilatura
from bs4 import BeautifulSoup
from tqdm import tqdm


def clean_text(text):
    """Membersihkan teks dari spasi berlebih dan teks pengganggu"""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def get_full_article_content(article_url, headers):
    """Mengambil isi berita lengkap menggunakan trafilatura dengan mode single-page (?single=1)"""
    try:
        if "?" in article_url:
            full_url = f"{article_url}&single=1"
        else:
            full_url = f"{article_url}?single=1"

        # 1. Download halaman web menggunakan trafilatura (atau requests)
        downloaded = trafilatura.fetch_url(full_url)

        if downloaded is None:
            # Fallback pakai requests biasa jika trafilatura fetch gagal
            res = requests.get(full_url, headers=headers, timeout=10)
            if res.status_code != 200:
                return None
            downloaded = res.text

        # 2. Ekstraksi isi artikel utama otomatis pakai trafilatura
        extracted_text = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=False,
            no_fallback=False,
        )

        if not extracted_text:
            return None

        # 3. Bersihkan teks hasil ekstraksi trafilatura
        full_content = clean_text(extracted_text)

        return full_content

    except Exception:
        return None


def get_detik_articles(index_url, label_name, total_target=100):
    articles = []
    page = 1

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
            " (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
    }

    print(f"=== Mulai mengambil data untuk label: {label_name} ===")

    # Menggunakan tqdm untuk progress bar visual
    pbar = tqdm(total=total_target, desc=f"Scraping {label_name.upper()}")

    while len(articles) < total_target:
        # Paginasi URL detik.com
        url = f"{index_url}?page={page}"
        res = requests.get(url, headers=headers)

        if res.status_code != 200:
            break

        soup = BeautifulSoup(res.text, "html.parser")
        list_articles = soup.find_all("article")

        if not list_articles:
            break

        for article in list_articles:
            if len(articles) >= total_target:
                break

            link_tag = article.find("a")
            if not link_tag or "href" not in link_tag.attrs:
                continue

            article_url = link_tag["href"]

            if "detik.com" not in article_url or "20" not in article_url:
                continue

            content = get_full_article_content(article_url, headers)

            if content and len(content) > 200:
                articles.append(content)
                pbar.update(1)

            time.sleep(0.2)

        page += 1

    pbar.close()
    return articles


# 1. Ambil 100 berita Sport & 100 berita Finance
sport_contents = get_detik_articles(
    "https://sport.detik.com/indeks", "sport", 100
)
finance_contents = get_detik_articles(
    "https://finance.detik.com/indeks", "finance", 100
)

# 2. Gabungkan data ke dalam satu struktur (3 Kolom: ID, Isi Berita, Label)
final_rows = []
current_id = 1

for content in sport_contents:
    final_rows.append(
        {"ID": current_id, "Isi Berita": content, "Label": "sport"}
    )
    current_id += 1

for content in finance_contents:
    final_rows.append(
        {"ID": current_id, "Isi Berita": content, "Label": "finance"}
    )
    current_id += 1

# Buat DataFrame pandas
df_all = pd.DataFrame(final_rows)
print("\nProses crawling selesai! Data siap disimpan.")

=== Mulai mengambil data untuk label: sport ===


Scraping SPORT: 100%|██████████| 100/100 [00:52<00:00,  1.89it/s]


=== Mulai mengambil data untuk label: finance ===


Scraping FINANCE: 100%|██████████| 100/100 [01:00<00:00,  1.65it/s]


Proses crawling selesai! Data siap disimpan.


Code tersebut berfungsi untuk melakukan web scraping artikel berita dari Detik.com dengan kategori "Sport" dan "Finance" dengan masing masing label berjumlah 100 artikel dengan memanfaatkan pustaka trafilatura untuk mengekstrak dan membersihkan teks artikel secara otomatis dari mode single-page, lalu menyusun seluruh data bersih tersebut kedalam sebuah DataFrame pandas yang rapi dengan 3 kolom (ID, Isi Berita< Label).

**Save & Download File Excel**

In [5]:
excel_filename = "berita_detik_200_artikel.xlsx"

# 1. Simpan ke Excel
df_all.to_excel(excel_filename, index=False)
print(f"File '{excel_filename}' berhasil dibuat!")

# 2. Tampilkan 5 baris pertama sebagai bukti
display(df_all.head())

File 'berita_detik_200_artikel.xlsx' berhasil dibuat!


,ID,Isi Berita,Label
0,1,Kejuaraan Nasional Gokart 2026 yang berlangsun...,sport
1,2,PT Bank Negara Indonesia (Persero) Tbk atau BN...,sport
2,3,"Pelatih Timnas voli putra Indonesia, Reidel To...",sport
3,4,Kontingen Indonesia bersiap menatap Asian Game...,sport
4,5,Rizki Juniansyah siap menjalani debut di Asian...,sport


Code diatas berfungsi untuk mengekspor DataFrame (df_all) menjadi file excel bernama "berita_detik_200_artikel.xlsx" tanpa sertaan indeks baris, kemudian menampilkan 5 baris pertama data sebagai pratinjau di antarmuka google colab, serta secara otomatis mengunduh file excel tersebut.